# 1. Command（节点内自决）

In [ ]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


# 定义状态
class State(MessagesState):
    pass


# llm_node 固定格式：读完整历史 -> 调模型 -> 返回消息增量 -> tool_node / END
def llm_node(state: State) -> Command[Literal["tool_node", END]]:
    ai_msg = model_with_tools.invoke(state["messages"])
    goto = "tool_node" if ai_msg.tool_calls else END
    return Command(goto=goto, update={"messages": [ai_msg]})


# tool_node 固定格式：遍历 tool_calls -> 逐个执行 -> 每个 call 一条 ToolMessage -> llm_node
def tool_node(state: State) -> Command["llm_node"]:
    last = state["messages"][-1]
    tool_map = {t.name: t for t in tools}  # 名字 -> 工具对象
    messages = []

    for tc in last.tool_calls:
        tool_name = tc["name"]
        tool_args = tc["args"]
        tool_id = tc["id"]
        selected = tool_map.get(tool_name)
        if selected:
            result = selected.invoke(tool_args)  # 调用工具函数
        else:
            result = f"未知工具{tool_name}"
        from langchain_core.messages import ToolMessage
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))
    return Command(goto="llm_node", update={"messages": messages})


builder = StateGraph(state_schema=State)

builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")

graph = builder.compile()

res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气和科技相关的新闻")]}, config=config)
print(res)

In [ ]:
from IPython.display import display

print(display(graph))

# 2. router + 条件边

In [ ]:
# 动态跳转 : Command [goto, update]
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print
from langchain_core.messages import ToolMessage

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: State) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


# 工具节点
def tool_node(state: State) -> State:
    last = state["messages"][-1]
    tool_map = {t.name: t for t in tools}
    messages = []
    for tc in last.tool_calls:
        selected = tool_map.get(tc["name"])
        if selected:
            res = selected.invoke(tc['args'])
        else:
            res = f"未知工具: {tc['name']}"

        messages.append(ToolMessage(content=res, tool_call_id=tc["id"]))

    return {"messages": messages}


def router(state: State) -> Literal["tool_node", END]:
    # 判断本轮模型输出：有 tool_calls 去工具节点，没有就结束
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])  # 动态跳转
builder.add_edge("tool_node", "llm_node")  # 固定回环

graph = builder.compile()
res = graph.invoke({"messages": [HumanMessage("帮我查一下北京的天气和科技相关的新闻")]}, config=config)
print(res)

In [ ]:
from IPython.display import display

print(display(graph))